# Kisan Mitra — Colab Development Notebook

**Before running:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

This notebook clones the project, installs dependencies, logs into Hugging Face, builds the retrieval index, and launches the live Gradio demo with a public `share=True` link. See `SETUP.md` in the repo for the one-time free account setup (Hugging Face token, optional data.gov.in API key).

In [ ]:
# 1. Clone the repo (replace with your actual GitHub URL once pushed)
REPO_URL = "https://github.com/<your-username>/kisan-mitra.git"
!git clone $REPO_URL project
%cd project

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Authenticate with Hugging Face (needed for the gated IndicTrans2 checkpoints)
from huggingface_hub import login
import getpass
login(getpass.getpass("Paste your Hugging Face token: "))

In [ ]:
# 4. (Optional) Set data.gov.in credentials to pull the full official KCC corpus.
# Leave blank to use the curated seed corpus only (still fully functional).
import os
os.environ["DATA_GOV_IN_API_KEY"] = ""  # optional
os.environ["KCC_RESOURCE_ID"] = ""      # optional
!python data/fetch_kcc_data.py

In [ ]:
# 5. Build the FAISS retrieval index over the corpus
!python rag/build_index.py

In [ ]:
# 6. Quick sanity check of the full pipeline before launching the UI
from pipeline import ask
result = ask("How do I control stem borer in paddy?", "English")
print(result.answer)
print("\n--- sources ---")
for s in result.sources:
    print(f"- {s['query']} (score={s['score']:.2f})")

In [ ]:
# 7. Launch the live demo (public link stays active while this cell keeps running)
from app import demo
demo.launch(share=True, debug=True)

## Running the evaluation harness

Run this in a separate cell/session once the index is built:
```python
!python eval/evaluate.py
```
This writes `eval/results.json` and `eval/failure_analysis.md`.